## Облачные графики на W&B

Создайте бесплатный аккаунт на `wandb.ai`, скопируйте API key из User Settings и добавьте его в Kaggle через **Add-ons → Secrets** под именем `WANDB_API_KEY`. Для notebook должен быть включён Internet. Ключ в код и output не выводится.

In [ ]:
# В свежей Kaggle-сессии ничего не обновляем: используем встроенный W&B.
import wandb
print("Preinstalled W&B version:", wandb.__version__)

Если предыдущая ячейка успешно импортировала W&B, выполняйте авторизацию ниже. Если import снова упал, перезапустите всю Kaggle Session. Не используйте `pip install -U` или `--force-reinstall`: они меняют общие зависимости Kaggle.

In [ ]:
import os
import wandb
from kaggle_secrets import UserSecretsClient

os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
assert wandb.login(key=os.environ["WANDB_API_KEY"], verify=True)
print("W&B authentication successful, version:", wandb.__version__)

# $S_7$: увеличенная модель и live TensorBoard

Этот запуск отделён от успешного протокола $S_6$. По сравнению с застрявшим вариантом уменьшена train-доля, увеличена ёмкость модели и batch, добавлены gradient clipping, EMA и TensorBoard с обновлением во время обучения.

In [ ]:
from pathlib import Path
import shutil, sys

EXPECTED_BUILD = "sn-stochastic-power-v1.3-wandb-2026-08-03"
candidates = [
    path for path in Path("/kaggle/input").rglob("train_sn_stochastic.py")
    if EXPECTED_BUILD in path.read_text(encoding="utf-8")
]
if not candidates:
    raise FileNotFoundError("Добавьте train_sn_stochastic.py версии v1.3 как Kaggle Input")
source = candidates[0]
local = Path("/kaggle/working/train_sn_stochastic.py")
shutil.copy2(source, local)
sys.path.insert(0, "/kaggle/working")
from train_sn_stochastic import BUILD_ID, Config, run
print("Script:", source)
print("Build:", BUILD_ID)
assert BUILD_ID == EXPECTED_BUILD

## Конфигурация

`f=0.03` оставляет примерно 762 тыс. train-пар. При batch 8192 это около 93 шагов на эквивалентную эпоху. Clipping срабатывает только на редких всплесках выше нормы 5 и отдельно логирует коэффициент срезания.

In [ ]:
CONFIG = Config(
    output_root="/kaggle/working/sn_stochastic_grokking",
    protocol_name="s7_live_e1024_h2048_f03_b8192_wd002",
    n_values=(7,),
    seeds=(42,),

    train_fraction=0.03,
    embedding_dim_by_n={7: 1024},
    hidden_dim_by_n={7: 2048},
    batch_size_by_n={7: 8192},
    max_steps_by_n={7: 500_000},

    learning_rate=7.5e-4,
    weight_decay=0.002,
    betas=(0.9, 0.98),
    warmup_steps=500,
    gradient_clip_norm=5.0,

    log_every=20,
    diagnostic_every=1_000,
    checkpoint_every=2_000,
    monitor_train_pairs=8192,
    monitor_val_pairs=8192,

    tensorboard=True,
    tensorboard_flush_secs=5,
    tensorboard_flush_every_logs=10,
    log_ema_beta=0.98,
    wandb_enabled=True,
    wandb_project="sn-grokking",
    wandb_entity=None,
    wandb_mode="online",

    train_threshold=0.99,
    val_threshold=0.95,
    patience_logs=10,
    required_gap_steps=30_000,
    post_grok_steps=15_000,

    use_amp=False,
    force_restart=True,
    resume_search_roots=("/kaggle/input",),
)
CONFIG

## Локальный TensorBoard (необязательно)

Основной live-dashboard теперь находится на W&B. Эту ячейку можно также выполнить для локальной панели TensorBoard; обе панели читают один поток метрик.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /kaggle/working/sn_stochastic_grokking --reload_interval 5

In [ ]:
RUN_DIRS = run(CONFIG)
RUN_DIRS

## Быстрая проверка CSV

После остановки или завершения запуска эта ячейка показывает последние точки. При возобновлении поставьте `force_restart=False` и не меняйте `protocol_name` или архитектуру.

In [ ]:
import pandas as pd
for run_dir in RUN_DIRS:
    frame = pd.read_csv(run_dir / "training_log.csv")
    display(frame[[
        "step", "epoch_equivalent", "monitor_train_loss",
        "monitor_train_acc", "val_loss", "val_acc",
        "ema_monitor_train_loss", "ema_monitor_train_acc",
        "ema_val_loss", "ema_val_acc", "grad_norm_preclip",
        "grad_clip_scale", "learning_rate"
    ]].tail(20))